# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# Exercise: Build a complete AI travel assistant that:
# 1. Uses function calling to retrieve ticket prices from a SQLite database.
# 2. Generates a city-specific pop-art illustration using an AI model.
# 3. Converts the generated SVG artwork into a PNG image for display.
# 4. Generates a voice response using a text-to-speech model.
# 5. Combines chat, tool calling, image generation, and audio playback
#    into an interactive Gradio application.

import os
os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/opt/cairo/lib"
import json
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from IPython.display import display, SVG
import base64
from io import BytesIO
from PIL import Image
import cairosvg

# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")
if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:8]}")
else:
    print("Google API Key not set")

if OPENROUTER_API_KEY and OPENROUTER_API_KEY.startswith("sk-or-"):
    print("OPENROUTER_API_KEY looks good so far")
else:
    print("OPENROUTER_API_KEY doesn't seem right")
    

ollama_Model = 'gpt-oss'
gemini_Model = 'gemini-3.1-flash-lite'
deepseek_Model = 'deepseek-ai/Janus-Pro-7B'
openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepinfra.com/v1/inference/"


gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

DB = "prices.db"


#system prompt
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


#get ticker price
def get_ticket_price(city):
	print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
	with sqlite3.connect(DB) as conn:
		cursor = conn.cursor()
		cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
		result = cursor.fetchone()
		return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"



#price function
price_function = {
	"name": "get_ticket_price",
	"description": "Get the price of a return ticket to the destination city.",
	"parameters": {
			"type": "object",
			"properties": {
					"destination_city": {
							"type": "string",
							"description": "The city that the customer wants to travel to",
					},
			},
			"required": ["destination_city"],
			"additionalProperties": False
	}
}
tools = [{"type": "function", "function": price_function}]
tools


#image-generation
def artist(city):
	prompt = f"""
	Create a vibrant pop-art SVG illustration of {city}, centered on its most iconic landmark or recognizable city scene.
	Use bold colors, clean vector shapes, and a fun tourist-poster style.
	Return only valid SVG markup starting with <svg and ending with </svg>.
	"""

	response = openrouter.chat.completions.create(
		model="deepseek/deepseek-v3.2",
		messages=[
			{
				"role": "user",
				"content": [
						{
								"type": "text",
								"text": prompt
						}
				]
			}
		]
	)
	result = response.choices[0].message.content
	#convert svg to png
	result = result.replace("```svg", "").replace("```", "").strip()
	png_data = cairosvg.svg2png(
			bytestring=result.encode("utf-8")
	)

	return Image.open(BytesIO(png_data))
	


#voice generation
def talker(message):
	response = openrouter.audio.speech.create(
		model="deepgram/flux-tts:free",
		input=message,
		voice="flux-alexis-en",
		response_format="mp3"
	)
	# return response.content
	with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
		f.write(response.content)
		return f.name


#chat function
def chat(history):
	history = [{"role":h["role"], "content":h["content"]} for h in history]
	messages = [{"role": "system", "content": system_message}] + history
	response = gemini.chat.completions.create(model=gemini_Model, messages=messages, tools=tools)
	cities = []
	image = None

	while response.choices[0].finish_reason=="tool_calls":
		message = response.choices[0].message
		responses, cities = handle_tool_calls_and_return_cities(message)
		messages.append(message)
		messages.extend(responses)
		response = gemini.chat.completions.create(model=gemini_Model, messages=messages, tools=tools)
	reply = response.choices[0].message.content
	history += [{"role":"assistant", "content":reply}]

	voice = talker(reply)
	print(f"reply:: {reply}")
	print(f"voice:: {voice}")
	if cities:
			image = artist(cities[0])
	
	return history, voice, image



#tool call
def handle_tool_calls_and_return_cities(message):
	responses = []
	cities = []
	for tool_call in message.tool_calls:
			if tool_call.function.name == "get_ticket_price":
					arguments = json.loads(tool_call.function.arguments)
					city = arguments.get('destination_city')
					cities.append(city)
					price_details = get_ticket_price(city)
					responses.append({
							"role": "tool",
							"content": price_details,
							"tool_call_id": tool_call.id
					})
	return responses, cities


# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
	return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
	with gr.Row():
		chatbot = gr.Chatbot(height=500, type="messages")
		image_output = gr.Image(height=500, interactive=False)
		# image_output = gr.HTML()
	with gr.Row():
		audio_output = gr.Audio(type="filepath", autoplay=True)
	with gr.Row():
		message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks
	message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
		chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
	)

ui.launch(inbrowser=True, auth=("haseeb", "12345678"))



	